# Taller 5. Pipeline modular para prensa: bloques, filetes, orden de lectura y artículos

**Qué produce:** la página con los bloques numerados en orden de lectura y coloreados por artículo, los filetes detectados, una tabla de bloques con su artículo y su posición en la lectura, y la estructura completa en JSON (página → artículos → bloques).

**Para qué:** pasar de los bloques sueltos de los Talleres 1 a 4 a la unidad que interesa en la prensa: el **artículo**. El cuaderno reproduce el enfoque «de abajo hacia arriba» de [Mocaër et al. (2026)](https://arxiv.org/html/2607.15082v1), que la Biblioteca Nacional de Francia prevé aplicar a su colección de diarios: cuatro etapas encadenadas, cada una visible y evaluable por separado.

| Etapa | En el artículo | En este cuaderno |
|---|---|---|
| 1. Bloques | YOLO26 entrenado con el diario *La Liberté* | DocLayout-YOLO (el del Taller 1), genérico |
| 2. Filetes | LSD (OpenCV) | LSD (OpenCV) |
| 3. Orden de lectura | LayoutReader reentrenado con *La Liberté* | LayoutReader original (`hantian/layoutreader`) |
| 4. Artículos | regla: cada título abre un artículo | la misma regla |

**Es una reconstrucción, no una réplica.** Los autores no publicaron sus modelos entrenados, así que las etapas 1 y 3 usan modelos genéricos. Los resultados no son comparables con los del artículo; lo que se puede observar es **cómo funciona el método** y **dónde falla cada etapa**.

**Entorno:** Colab con GPU T4 recomendada (*Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*). Funciona también sin GPU, más lento. No hace OCR: para leer el texto de cada bloque, ver los Talleres 2 a 4 o el Taller 6.

## 0. Instalar dependencias (solo Colab)

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "doclayout-yolo==0.0.4", "pypdfium2"], check=True)
    print("Instalado doclayout-yolo 0.0.4; transformers y OpenCV ya vienen en Colab")
else:
    print("Fuera de Colab no se instala nada: preparar el entorno según cuadernos/README.md.")

## 1. Elegir las imágenes

Por defecto se procesa la **página común del taller** (*Centro Gallego*, Montevideo, 1917): la misma página en todos los cuadernos permite comparar modelos. Cambiá `SELECCION` para trabajar con otras fuentes:

| Valor | Qué procesa |
|---|---|
| `"comun"` | la página común (recomendado para la primera pasada) |
| `"prensa"` | todas las páginas de prensa del taller |
| `"manuscritos"` | cartas, actas y otros manuscritos del taller |
| `"todas"` | prensa y manuscritos |
| `"propias"` | tus imágenes o PDF; en Colab aparecen dos botones para subir **varios archivos** o **una carpeta completa**; fuera de Colab se leen de `mis_imagenes/` (incluidas sus subcarpetas) |

Cada página de un PDF se convierte en una imagen PNG (resolución `DPI_PDF = 200`) y se procesa como una página más.

**No subas material sensible o con restricciones de uso** a servicios externos.

In [ ]:
SELECCION = "comun"

In [ ]:
import csv
import sys
import urllib.request
import uuid
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display
from PIL import Image

EN_COLAB = "google.colab" in sys.modules
REPO_RAW = "https://raw.githubusercontent.com/agusnieto77/sicss-uruguay-2026/main/cuadernos/imagenes/"
BASE = Path("/content/taller") if EN_COLAB else Path.cwd()
CARPETA_IMAGENES = BASE / "imagenes"
EXTENSIONES_IMAGEN = {".png", ".jpg", ".jpeg", ".webp", ".tif", ".tiff"}
EXTENSIONES = EXTENSIONES_IMAGEN | {".pdf"}
DPI_PDF = 200  # resolución con la que cada página de un PDF se convierte en imagen


def traer(relativa):
    """Ruta local de un archivo del taller; en Colab lo descarga del repositorio la primera vez."""
    destino = CARPETA_IMAGENES / relativa
    if EN_COLAB and not destino.exists():
        destino.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(REPO_RAW + relativa, destino)
    if not destino.exists():
        raise FileNotFoundError(f"No encuentro {destino}. Abrí el cuaderno desde la carpeta cuadernos/ del repositorio.")
    return destino


def mostrar(imagen, ancho=1000):
    """Muestra una imagen PIL reducida a un ancho cómodo para la pantalla."""
    if imagen.width > ancho:
        imagen = imagen.resize((ancho, round(imagen.height * ancho / imagen.width)))
    display(imagen)


def admitido(nombre):
    """Imágenes y PDF, sin archivos ocultos (por ejemplo, los `._foto.jpg` que deja macOS en las carpetas)."""
    return not Path(nombre).name.startswith(".") and Path(nombre).suffix.lower() in EXTENSIONES


def pdf_a_imagenes(pdf, carpeta):
    """Convierte cada página de un PDF en una imagen PNG y devuelve sus rutas."""
    import pypdfium2 as pdfium

    carpeta.mkdir(parents=True, exist_ok=True)
    documento = pdfium.PdfDocument(pdf)
    paginas = []
    for indice in range(len(documento)):
        pagina = documento[indice]
        destino = carpeta / f"{pdf.stem}_p{indice + 1:03d}.png"
        pagina.render(scale=DPI_PDF / 72).to_pil().convert("RGB").save(destino)
        pagina.close()
        paginas.append(destino)
    documento.close()
    return paginas


SELECTOR_COLAB = """
<div style="display:flex; gap:2em; align-items:center; flex-wrap:wrap">
  <label><b>Elegir archivos</b> (imágenes y PDF):
    <input type="file" id="__ID__-archivos" multiple accept="image/*,.tif,.tiff,.pdf"></label>
  <span><b>o una carpeta completa</b>:
    <input type="file" id="__ID__-carpeta" webkitdirectory style="display:none"
      onchange="document.getElementById('__ID__-estado').textContent = this.files.length
        ? this.files[0].webkitRelativePath.split('/')[0] + ' (' + this.files.length + ' archivos)' : ''">
    <button type="button" onclick="document.getElementById('__ID__-carpeta').click()">Seleccionar carpeta</button>
    <span id="__ID__-estado">Ninguna carpeta seleccionada</span></span>
</div>
<script>
window.subidaTaller = {
  elegir: (id) => new Promise((resolve) => {
    for (const sufijo of ["archivos", "carpeta"]) {
      document.getElementById(id + "-" + sufijo).addEventListener("change", (evento) => {
        window.subidaTaller.lista = Array.from(evento.target.files);
        resolve(window.subidaTaller.lista.map((f) => f.webkitRelativePath || f.name));
      });
    }
  }),
  trozo: (indice, inicio, fin) => new Promise((resolve, reject) => {
    const lector = new FileReader();
    lector.onload = () => resolve(lector.result.split(",")[1] || "");
    lector.onerror = () => reject(lector.error);
    lector.readAsDataURL(window.subidaTaller.lista[indice].slice(inicio, fin));
  }),
  tamano: (indice) => window.subidaTaller.lista[indice].size,
};
</script>
"""


def subir_en_colab(carpeta, trozo=1024 * 1024):
    """Diálogo de Colab para elegir varios archivos o una carpeta; guarda imágenes y PDF en `carpeta`."""
    import base64

    from google.colab import output

    identificador = "subida-" + uuid.uuid4().hex
    display(HTML(SELECTOR_COLAB.replace("__ID__", identificador)))
    nombres = output.eval_js(f'window.subidaTaller.elegir("{identificador}")')
    for indice, nombre in enumerate(nombres):
        if not admitido(nombre):
            print("Ignorado (no es imagen ni PDF):", nombre)
            continue
        destino = carpeta.joinpath(*Path(nombre).parts)  # una carpeta conserva sus subcarpetas
        destino.parent.mkdir(parents=True, exist_ok=True)
        tamano = output.eval_js(f"window.subidaTaller.tamano({indice})")
        with open(destino, "wb") as archivo:
            for inicio in range(0, tamano, trozo):
                datos = output.eval_js(f"window.subidaTaller.trozo({indice}, {inicio}, {inicio + trozo})")
                archivo.write(base64.b64decode(datos))
        print(f"Subido: {nombre} ({tamano / 1e6:.1f} MB)")


FICHAS = {}
if SELECCION == "propias":
    carpeta_propia = BASE / "mis_imagenes"
    carpeta_propia.mkdir(parents=True, exist_ok=True)
    if EN_COLAB:
        subir_en_colab(carpeta_propia)
    IMAGENES = []
    for archivo in sorted(p for p in carpeta_propia.rglob("*") if p.is_file() and admitido(p.name)):
        if archivo.suffix.lower() == ".pdf":
            paginas = pdf_a_imagenes(archivo, BASE / "paginas_pdf")
            print(f"{archivo.name}: {len(paginas)} página(s) convertidas a imagen a {DPI_PDF} dpi")
            IMAGENES += paginas
        else:
            IMAGENES.append(archivo)
else:
    with open(traer("fuentes.csv"), encoding="utf-8-sig") as archivo:
        fichas = list(csv.DictReader(archivo, delimiter=";"))
    filtros = {
        "comun": lambda f: f["pagina_comun"] == "si",
        "prensa": lambda f: f["tipo"] == "prensa",
        "manuscritos": lambda f: f["tipo"] != "prensa",
        "todas": lambda f: True,
    }
    if SELECCION not in filtros:
        raise ValueError(f"SELECCION debe ser uno de {sorted(filtros) + ['propias']}")
    elegidas = [f for f in fichas if filtros[SELECCION](f)]
    IMAGENES = [traer(f["archivo"]) for f in elegidas]
    FICHAS = {ruta.name: ficha for ruta, ficha in zip(IMAGENES, elegidas)}
    display(pd.DataFrame(elegidas)[["archivo", "tipo", "titulo", "fecha", "lugar", "institucion", "licencia"]])

assert IMAGENES, "No hay imágenes para procesar: revisá SELECCION o la carpeta mis_imagenes/."
print(f"{len(IMAGENES)} imagen(es):", [p.name for p in IMAGENES])

## 2. Cargar los modelos

- **DocLayout-YOLO** detecta y clasifica los bloques. `CONFIANZA = 0.20` es el umbral que usa el artículo.
- **LayoutReader** predice el orden de lectura. Es un transformer de la familia LayoutLM (LayoutLMv3-large ajustado) que recibe **solo las coordenadas** de los bloques: no ve la imagen ni lee el texto. Se entrenó con fragmentos de línea del corpus ReadingBank, no con bloques de prensa. Licencia CC BY-NC-SA 4.0: uso no comercial y con atribución.

In [ ]:
import torch
from doclayout_yolo import YOLOv10
from huggingface_hub import hf_hub_download
from transformers import LayoutLMv3ForTokenClassification

DISPOSITIVO = "cuda:0" if torch.cuda.is_available() else "cpu"
CONFIANZA = 0.20
RESOLUCION = 1024

detector = YOLOv10(hf_hub_download(repo_id="juliozhao/DocLayout-YOLO-DocStructBench", filename="doclayout_yolo_docstructbench_imgsz1024.pt"))
lector = LayoutLMv3ForTokenClassification.from_pretrained("hantian/layoutreader").to(DISPOSITIVO).eval()

CLASES_ES = {
    "title": "título",
    "plain text": "texto",
    "abandon": "descartable: encabezado, pie, número de página, marginalia",
    "figure": "figura o grabado",
    "figure_caption": "epígrafe de figura",
    "table": "tabla",
    "table_caption": "título de tabla",
    "table_footnote": "nota de tabla",
    "isolate_formula": "fórmula",
    "formula_caption": "número de fórmula",
}
print("Dispositivo:", DISPOSITIVO)

## 3. Etapa 1: detectar bloques

DocLayout-YOLO a veces propone dos cajas casi superpuestas para la misma zona. Como en el artículo, se eliminan las superposiciones: si una caja queda contenida en más de `SUPERPOSICION_MAXIMA` de su superficie dentro de otra con mayor confianza, se descarta.

In [ ]:
SUPERPOSICION_MAXIMA = 0.8


def sin_superposiciones(cajas):
    """Descarta las cajas contenidas casi por completo en otra de mayor confianza."""
    conservadas = []
    for caja in sorted(cajas, key=lambda c: c["confianza"], reverse=True):
        area = (caja["x2"] - caja["x1"]) * (caja["y2"] - caja["y1"])
        if all(max(0, min(caja["x2"], otra["x2"]) - max(caja["x1"], otra["x1"])) * max(0, min(caja["y2"], otra["y2"]) - max(caja["y1"], otra["y1"]))
               < SUPERPOSICION_MAXIMA * area for otra in conservadas):
            conservadas.append(caja)
    return conservadas


def detectar_bloques(pagina, ancho, alto):
    """Bloques detectados por DocLayout-YOLO, con su clase y su confianza."""
    resultado = detector.predict(str(pagina), imgsz=RESOLUCION, conf=CONFIANZA, device=DISPOSITIVO, verbose=False)[0]
    bloques = []
    for xyxy, clase, confianza in zip(resultado.boxes.xyxy.cpu().numpy(), resultado.boxes.cls.cpu().numpy(), resultado.boxes.conf.cpu().numpy()):
        x1, y1, x2, y2 = (int(v) for v in xyxy)
        x1, y1, x2, y2 = max(0, x1), max(0, y1), min(ancho, x2), min(alto, y2)
        if x2 > x1 and y2 > y1:
            bloques.append({"clase": detector.names[int(clase)], "confianza": round(float(confianza), 3), "x1": x1, "y1": y1, "x2": x2, "y2": y2})
    return sin_superposiciones(bloques)

## 4. Etapa 2: detectar filetes

Los diarios separan columnas y artículos con **filetes**: líneas impresas horizontales y verticales. LSD (*Line Segment Detector*) es un algoritmo clásico de visión por computadora, sin entrenamiento: encuentra todos los segmentos rectos de la imagen. El cuaderno conserva solo los largos y casi horizontales o verticales.

- `LARGO_MINIMO`: fracción del ancho (o del alto) de la página que debe medir un segmento para contar como filete.
- `INCLINACION_MAXIMA`: desvío tolerado respecto de la horizontal o la vertical.

En el artículo, los filetes **entran a LayoutReader** como información adicional, porque los autores lo reentrenaron para usarlos. El LayoutReader original no sabe hacerlo: aquí los filetes se muestran, se exportan y pueden usarse en la etapa 4 para separar artículos (`CORTAR_CON_FILETES`).

In [ ]:
import cv2
import numpy as np

LARGO_MINIMO = 0.08
INCLINACION_MAXIMA = 0.03


def detectar_filetes(imagen):
    """Segmentos rectos largos (filetes) detectados con LSD."""
    gris = cv2.cvtColor(np.array(imagen), cv2.COLOR_RGB2GRAY)
    segmentos = cv2.createLineSegmentDetector().detect(gris)[0]
    filetes = []
    for x1, y1, x2, y2 in ([] if segmentos is None else segmentos.reshape(-1, 4)):
        dx, dy = abs(x2 - x1), abs(y2 - y1)
        if dy <= dx * INCLINACION_MAXIMA and dx >= LARGO_MINIMO * imagen.width:
            orientacion = "horizontal"
        elif dx <= dy * INCLINACION_MAXIMA and dy >= LARGO_MINIMO * imagen.height:
            orientacion = "vertical"
        else:
            continue
        filetes.append({
            "orientacion": orientacion,
            "x1": round(float(min(x1, x2))), "y1": round(float(min(y1, y2))),
            "x2": round(float(max(x1, x2))), "y2": round(float(max(y1, y2))),
        })
    return filetes

## 5. Etapa 3: orden de lectura con LayoutReader

Como en el artículo, los bloques entran preordenados de arriba hacia abajo y de izquierda a derecha, con las coordenadas llevadas a una escala de 0 a 1000. Para cada posición de la secuencia de lectura, el modelo puntúa qué bloque debería ocuparla. Si dos posiciones eligen el mismo bloque, se lo queda la más segura y la otra pasa a su siguiente opción. El modelo admite hasta 510 bloques por página.

In [ ]:
MAX_BLOQUES = 510


def ordenar_con_layoutreader(bloques, ancho, alto):
    """Devuelve los bloques en el orden de lectura que predice LayoutReader."""
    bloques = sorted(bloques, key=lambda b: (b["y1"], b["x1"]))
    if not bloques:
        return []
    if len(bloques) > MAX_BLOQUES:
        raise ValueError(f"{len(bloques)} bloques: LayoutReader admite hasta {MAX_BLOQUES}. Subir CONFIANZA.")
    cajas = [[round(b["x1"] * 1000 / ancho), round(b["y1"] * 1000 / alto), round(b["x2"] * 1000 / ancho), round(b["y2"] * 1000 / alto)] for b in bloques]
    n = len(cajas)
    entrada = {
        "bbox": torch.tensor([[[0, 0, 0, 0]] + cajas + [[0, 0, 0, 0]]]),
        "input_ids": torch.tensor([[0] + [3] * n + [2]]),  # sin texto: cada bloque es un símbolo «desconocido»
        "attention_mask": torch.ones(1, n + 2, dtype=torch.long),
    }
    with torch.no_grad():
        puntajes = lector(**{k: v.to(lector.device) for k, v in entrada.items()}).logits[0, 1:n + 1, :n].cpu()
    preferencias = puntajes.argsort(dim=1, descending=True).tolist()
    elegido = [opciones.pop(0) for opciones in preferencias]  # elegido[posición] = índice del bloque
    while True:
        usos = {}
        for lugar, indice in enumerate(elegido):
            usos.setdefault(indice, []).append(lugar)
        choques = [lugares for lugares in usos.values() if len(lugares) > 1]
        if not choques:
            break
        for lugares in choques:
            indice = elegido[lugares[0]]
            ganador = max(lugares, key=lambda lugar: puntajes[lugar, indice])
            for lugar in lugares:
                if lugar != ganador:
                    elegido[lugar] = preferencias[lugar].pop(0)
    return [bloques[indice] for indice in elegido]

## 6. Etapa 4: agrupar en artículos

La regla del artículo: se recorren los bloques en orden de lectura y **cada título (o grupo de títulos seguidos) abre un artículo nuevo**. Los bloques anteriores al primer título forman un artículo sin título. Los bloques `abandon` (encabezados, pies, números de página) quedan fuera de los artículos, con `articulo = 0`.

`CORTAR_CON_FILETES = True` agrega una regla que no está en el artículo: también se abre un artículo nuevo cuando un filete horizontal separa dos bloques consecutivos de la misma columna.

In [ ]:
CORTAR_CON_FILETES = False
TOLERANCIA = 10  # píxeles


def hay_filete_entre(arriba, abajo, filetes):
    """True si un filete horizontal separa dos bloques consecutivos de la misma columna."""
    if abajo["y1"] < arriba["y2"] - TOLERANCIA:
        return False
    for filete in filetes:
        if filete["orientacion"] != "horizontal":
            continue
        if not arriba["y2"] - TOLERANCIA <= filete["y1"] <= abajo["y1"] + TOLERANCIA:
            continue
        cruce = min(filete["x2"], abajo["x2"]) - max(filete["x1"], abajo["x1"])
        if cruce >= 0.5 * (abajo["x2"] - abajo["x1"]):
            return True
    return False


def agrupar_en_articulos(ordenados, filetes):
    """Asigna a cada bloque un número de artículo recorriendo la página en orden de lectura."""
    articulo, anterior = 0, None
    for bloque in ordenados:
        if bloque["clase"] == "abandon":
            bloque["articulo"] = 0
            continue
        nuevo = anterior is None or (bloque["clase"] == "title" and anterior["clase"] != "title")
        if CORTAR_CON_FILETES and anterior is not None and hay_filete_entre(anterior, bloque, filetes):
            nuevo = True
        articulo += nuevo
        bloque["articulo"] = articulo
        anterior = bloque
    return ordenados

## 7. Procesar las páginas

En la imagen, cada color es un artículo y cada etiqueta indica `orden · artículo`. La línea gris une los bloques en el orden de lectura predicho; los filetes aparecen en magenta. Los bloques fuera de artículos se dibujan en gris.

In [ ]:
import json

from PIL import ImageDraw, ImageFont

SALIDA = BASE / "resultados" / "05_pipeline_modular"
SALIDA.mkdir(parents=True, exist_ok=True)
COLORES = ["#e63946", "#1d3557", "#2a9d8f", "#e9c46a", "#8338ec", "#f4a261", "#06d6a0", "#ef476f", "#118ab2", "#bc6c25"]


def dibujar(imagen, ordenados, filetes):
    vista = imagen.copy()
    lapiz = ImageDraw.Draw(vista)
    grosor = max(3, imagen.width // 500)
    fuente = ImageFont.load_default(size=max(16, imagen.width // 90))
    for filete in filetes:
        lapiz.line((filete["x1"], filete["y1"], filete["x2"], filete["y2"]), fill="#ff00ff", width=grosor)
    centros = [((b["x1"] + b["x2"]) / 2, (b["y1"] + b["y2"]) / 2) for b in ordenados]
    if len(centros) > 1:
        lapiz.line(centros, fill="#555555", width=max(2, grosor // 2))
    for bloque in ordenados:
        color = "#6c757d" if bloque["articulo"] == 0 else COLORES[(bloque["articulo"] - 1) % len(COLORES)]
        lapiz.rectangle((bloque["x1"], bloque["y1"], bloque["x2"], bloque["y2"]), outline=color, width=grosor)
        lapiz.text((bloque["x1"] + grosor, bloque["y1"] + grosor), f"{bloque['orden']} · A{bloque['articulo']}",
                   fill=color, font=fuente, stroke_width=3, stroke_fill="white")
    return vista


filas, estructura = [], []
for pagina in IMAGENES:
    imagen = Image.open(pagina).convert("RGB")
    bloques = detectar_bloques(pagina, imagen.width, imagen.height)
    filetes = detectar_filetes(imagen)
    ordenados = ordenar_con_layoutreader(bloques, imagen.width, imagen.height)
    for orden, bloque in enumerate(ordenados, 1):
        bloque["orden"] = orden
    agrupar_en_articulos(ordenados, filetes)

    destino = SALIDA / pagina.stem
    destino.mkdir(parents=True, exist_ok=True)
    vista = dibujar(imagen, ordenados, filetes)
    vista.save(destino / "articulos.png")
    articulos = sorted({b["articulo"] for b in ordenados} - {0})
    print(f"{pagina.name}: {len(ordenados)} bloques, {len(articulos)} artículos, "
          f"{sum(f['orientacion'] == 'horizontal' for f in filetes)} filetes horizontales y {sum(f['orientacion'] == 'vertical' for f in filetes)} verticales")
    mostrar(vista)

    for bloque in ordenados:
        filas.append({"pagina": pagina.name, **bloque, "clase_es": CLASES_ES.get(bloque["clase"], ""), "evaluacion_humana": ""})
    estructura.append({
        "pagina": pagina.name,
        "ancho": imagen.width,
        "alto": imagen.height,
        "articulos": [{"articulo": a, "bloques": [b for b in ordenados if b["articulo"] == a]} for a in articulos],
        "fuera_de_articulos": [b for b in ordenados if b["articulo"] == 0],
        "filetes": filetes,
    })

COLUMNAS = ["pagina", "orden", "articulo", "clase", "clase_es", "confianza", "x1", "y1", "x2", "y2", "evaluacion_humana"]
bloques = pd.DataFrame(filas, columns=COLUMNAS)
bloques.to_csv(SALIDA / "bloques.csv", index=False, sep=";", decimal=",", encoding="utf-8-sig")
(SALIDA / "estructura.json").write_text(json.dumps(estructura, ensure_ascii=False, indent=2), encoding="utf-8")
print("Guardado en:", SALIDA)

## 8. Artículos reconstruidos

La tabla resume cada artículo: cuántos bloques tiene, de qué clases y qué superficie ocupa. Debajo, el recorte de los primeros artículos de la primera página: la caja que envuelve todos sus bloques, que puede incluir partes de otros artículos si el artículo ocupa varias columnas.

In [ ]:
import matplotlib.pyplot as plt

en_articulos = bloques[bloques["articulo"] > 0]
if en_articulos.empty:
    print("No se reconstruyeron artículos con CONFIANZA =", CONFIANZA)
else:
    resumen = en_articulos.groupby(["pagina", "articulo"]).agg(
        bloques=("orden", "count"),
        clases=("clase", lambda c: ", ".join(c)),
        x1=("x1", "min"), y1=("y1", "min"), x2=("x2", "max"), y2=("y2", "max"),
    ).reset_index()
    pd.set_option("display.max_colwidth", 120)
    display(resumen)

    primera = resumen[resumen["pagina"] == resumen["pagina"].iloc[0]].head(8)
    original = Image.open(next(p for p in IMAGENES if p.name == primera["pagina"].iloc[0])).convert("RGB")
    figura, ejes = plt.subplots(2, 4, figsize=(16, 10))
    for eje in ejes.flat:
        eje.axis("off")
    for eje, (_, fila) in zip(ejes.flat, primera.iterrows()):
        eje.imshow(original.crop((fila["x1"], fila["y1"], fila["x2"], fila["y2"])))
        eje.set_title(f"Artículo {fila['articulo']} · {fila['bloques']} bloques", fontsize=10)
    plt.tight_layout()
    plt.show()

## 9. Evaluar

Abrí `bloques.csv` (se abre directamente en Excel) y completá `evaluacion_humana` con `correcto`, `orden_erroneo`, `articulo_erroneo` (el bloque pertenece a otro artículo), `clase_erronea` u `omite` (agregá una fila para lo que no se detectó).

**Preguntas para la discusión**

1. ¿Los artículos reconstruidos coinciden con las noticias de la página? ¿Qué pasa con los avisos, las secciones fijas y los textos sin título?
2. La regla de los títulos depende de la etapa 1: ¿qué ocurre cuando el detector no reconoce un título, o clasifica como título una línea destacada dentro de un texto?
3. Activá `CORTAR_CON_FILETES = True` y volvé a ejecutar desde la etapa 4. ¿Los filetes separan artículos, o también marcos de avisos y grabados?
4. LayoutReader no ve la imagen ni el texto: ¿qué errores de orden se explican por eso? Compará con el orden del Taller 2 (reglas) y del Taller 3 (orden aprendido).
5. En el artículo, el YOLO y el LayoutReader se reentrenaron con miles de páginas del mismo diario. ¿Qué haría falta para hacer lo mismo con la prensa del Río de la Plata?

## Registro de mediaciones

Toda transcripción automática depende de la herramienta, la versión, la configuración y el equipo que la produjeron. Esta celda guarda esos datos en `procesamiento.json`, junto a los resultados, para poder citarlos y reproducirlos.

In [ ]:
import datetime
import importlib.metadata as metadata
import json
import platform

registro = {
    "herramienta": "Pipeline modular de abajo hacia arriba (reconstrucción de Mocaër et al., 2026)",
    "fecha": datetime.datetime.now().isoformat(timespec="seconds"),
    "entorno": "Google Colab" if EN_COLAB else platform.platform(),
    "python": platform.python_version(),
    "dispositivo": DISPOSITIVO,
    "imagenes": [p.name for p in IMAGENES],
    "versiones": {**{p: metadata.version(p) for p in ("doclayout-yolo", "torch", "transformers")}, "opencv": cv2.__version__},
    "modelos": {
        "bloques": "juliozhao/DocLayout-YOLO-DocStructBench (doclayout_yolo_docstructbench_imgsz1024.pt)",
        "filetes": "LSD (OpenCV)",
        "orden": "hantian/layoutreader",
        "articulos": "regla: cada título o grupo de títulos abre un artículo",
    },
    "parametros": {
        "confianza": CONFIANZA, "resolucion": RESOLUCION, "superposicion_maxima": SUPERPOSICION_MAXIMA,
        "largo_minimo_filete": LARGO_MINIMO, "inclinacion_maxima_filete": INCLINACION_MAXIMA,
        "cortar_con_filetes": CORTAR_CON_FILETES,
    },
}
(SALIDA / "procesamiento.json").write_text(json.dumps(registro, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(registro, ensure_ascii=False, indent=2))

## Descargar resultados

In [ ]:
import shutil

if EN_COLAB:
    from google.colab import files

    files.download(shutil.make_archive(f"/content/resultados_05_pipeline_modular", "zip", SALIDA))
else:
    print("Resultados en:", SALIDA.resolve())